In [1]:
import pandas as pd
import numpy as np
import os
import sys
%load_ext autoreload
%autoreload 2
%matplotlib widget
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from icu_sleep_breathing_2020_help_functions import * 

sys.path.append('/home/wolfgang/repos/sleep_research_io/')
from sleep_research_functions import *

pd.options.display.max_rows = 300
pd.options.display.max_columns = 300


font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 8}

font_headers =  {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 9}

font_subplots =  {'family' : 'sans-serif',
        'weight' : 'bold',
        'size'   : 9}

matplotlib.rc('font', **font)

In [2]:
plots_savedir = '/scr/wolfgang/Dropbox (Partners HealthCare)/SleepInICUandSleeplabPaper/Plots/hypnograms'
if not os.path.exists(plots_savedir):
    os.makedirs(plots_savedir)

In [3]:
[summary_subjects_icu, summary_subjects_sleeplab, 
 summary_days_icu, summary_days_sleeplab] = load_summary_data_with_inclusion_criteria()

summary_subjects_sleeplab = summary_subjects_sleeplab.query("matched_all == 1")
summary_subjects_sleeplab['sex'] = (summary_subjects_sleeplab['sex'] == 'Male').astype(int)

summary_days_sleeplab = summary_days_sleeplab.query("matched_all == 1")
summary_days_sleeplab['sex'] = (summary_days_sleeplab['sex'] == 'Male').astype(int)

/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3338: DtypeWarning: Columns (15,91,92,93,6455,6456,6457,12819,12820,12821,19185,19186,19187,21491,21492,21493,21519,21520,21521,21526,21527,21528,25549,25550,25551,31913,31914,31915,38279,38280,38281,44643,44644,44645,46851,46852,46853,46865,46866,46867,46879,46880,46881,46949,46950,46951,46956,46957,46958,46963,46964,46965,46970,46971,46972,46977,46978,46979,46984,46985,46986,51007,51008,51009,53336,53337,53338,57305,57373,57374,57375,59588,59589,59590,59602,59603,59604,59686,59687,59688,63669,63737,63738,63739,65952,65953,65954,65959,65960,65961,65980,65981,65982,66036,66037,66038,66043,66044,66045,66050,66051,66052,66057,66058,66059,66071,66072,66073,66078,66079,66080,70033,70101,70102,70103,72318,72319,72320,72332,72333,72334,72402,72403,72404,72416,72417,72418,76399,76467,76468,76469,78682,78683,78684,78696,78697,78698,78710,78711,78712,78766,78767,78768,78787,78788,78789,78801,7880

# of subjects ICU before inclusion criteria: 108
# of 12-hour segments ICU before inclusion criteria: 1230
# of subjects ICU after inclusion criteria: 102
# of 12-hour segments ICU after inclusion criteria: 1161
24-hour segments: 387
12-hour segments: 774

# of subjects sleeplab before inclusion criteria: 412
# of 12-hour segments sleeplab before inclusion criteria: 412


In [4]:
for agreement in ['all', 'agreement_relaxed', 'disagreement_relaxed', 'agreement', 'disagreement']:
    for modality in ['amendsumeffort', 'ecg_nn']:
        var_n2 = 'perc_N2_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)
        var_n3 = 'perc_N3_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)
        var_sum = 'perc_N2N3_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)

        summary_days_icu[var_sum] = summary_days_icu[var_n2] + summary_days_icu[var_n3]
        summary_days_sleeplab[var_sum] = summary_days_sleeplab[var_n2] + summary_days_sleeplab[var_n3]
        
        
for agreement in ['agreement_relaxed', 'agreement', 'all']:
    for modality in ['amendsumeffort', 'ecg_nn']:
        var_sleep = 'hours_sleep_MODALITY_all'.replace('MODALITY', modality) # all overlap sleep
        var_agreeing = 'hours_sleep_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement)
        
        var_discordant = 'hours_discordant_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement) # discordant sleep among overlap sleep
        summary_days_icu[var_discordant] = summary_days_icu[var_sleep] - summary_days_icu[var_agreeing]
        summary_days_sleeplab[var_discordant] = summary_days_sleeplab[var_sleep] - summary_days_sleeplab[var_agreeing]
        
        var_discordant_perc = 'perc_discordant_MODALITY_AGREEMENT'.replace('MODALITY', modality).replace('AGREEMENT', agreement) # discordant sleep among overlap sleep in %
        summary_days_icu[var_discordant_perc] = np.divide(summary_days_icu[var_sleep] - summary_days_icu[var_agreeing], summary_days_icu[var_sleep])
        summary_days_sleeplab[var_discordant_perc] = np.divide(summary_days_sleeplab[var_sleep] - summary_days_sleeplab[var_agreeing], summary_days_sleeplab[var_sleep])
        

In [5]:
dir_icu = f'/scr/wolfgang/Sleep_And_Breathing/icu_files_step6'
dir_sleeplab = f'/scr/wolfgang/Sleep_And_Breathing/sleeplab_files_step6'
print(os.listdir(dir_icu)[:2])
print(os.listdir(dir_sleeplab)[:2])

staging_cols_icu = ['stage_pred_ecg_nn', 'stage_pred_amendsumeffort'] 
staging_cols_sleeplab = [ 'stage_pred_expert', 'stage_pred_ecg_nn', 'stage_pred_amendsumeffort'] 
var_discordance = 'perc_discordant_ecg_nn_agreement_relaxed'


['icusleep_173.h5', 'icusleep_139.h5']
['psg_airgo_10hz_313.h5', 'psg_airgo_10hz_283.h5']


In [6]:
age_min = 45
age_max = 110
# sex = 0

# summary_days_icu = summary_days_icu.query("day_cat == 'f'")
# sel1_icu = summary_days_icu.query(f"age >= {age_min} & age <= {age_max} & sex == {sex}")
# sel1_sleeplab = summary_days_sleeplab.query(f"age >= {age_min} & age <= {age_max} & sex == {sex}")

summary_days_icu = summary_days_icu.query("day_cat == 'f'")
sel1_icu = summary_days_icu.query(f"age >= {age_min} & age <= {age_max}")
sel1_sleeplab = summary_days_sleeplab.query(f"age >= {age_min} & age <= {age_max}")


sel1_icu = sel1_icu.query(f"{var_discordance} > 0")
sel1_sleeplab = sel1_sleeplab.query(f"{var_discordance} > 0")
sel1_icu.sort_values(by=var_discordance, ascending=True, inplace=True)
sel1_sleeplab.sort_values(by=var_discordance, ascending=True, inplace=True)

print(f"#ICU: {sel1_icu.mrn.unique().shape[0]}")
print(f"#Sleeplab: {sel1_sleeplab.mrn.unique().shape[0]}")

colors = {
    'stage_pred_ecg_nn': 'crimson',
    'stage_pred_amendsumeffort': 'blue',
    'stage_pred_expert': 'orange',
}


#ICU: 79
#Sleeplab: 204


In [7]:
plt.close('all')
%matplotlib inline

for jloc, row in sel1_icu.iterrows():
    
    try:
        filepath = os.path.join(dir_icu, 'icusleep_' + str(row.study_id).zfill(3) + '.h5')
        data, hdr, fs = read_in_routine(filepath)

        print(f"discordance (%): {row[var_discordance]}")

        dt_start = row['timerange'].split(" - ")[0]
        dt_end = row['timerange'].split(" - ")[1]

        data = data.loc[dt_start : dt_end]


        fig, ax = plt.subplots(1, 1, figsize = (7.2, 3))

        for i_version, staging_version in enumerate(staging_cols_icu):
            ax.plot(data[staging_version] + i_version * 0.04, c=colors[staging_version])
            ax.set_title(f'study_id {row.study_id}, sex {row.sex}, age {np.round(row.age)} discordance: {np.round(row[var_discordance], 2)}')
            ax.set_yticks([5, 4, 3, 2, 1])
            ax.set_yticklabels(['W', 'R', 'N1', 'N2', 'N3'])

        plt.savefig(os.path.join(plots_savedir, f"icu_{row.study_id}_{dt_start}.png"),dpi=300)
        plt.savefig(os.path.join(plots_savedir, f"icu_{row.study_id}_{dt_start}.png"), dpi=300)
        
        plt.show()

    except Exception as e:
        print(e)
        continue

In [9]:
plt.close('all')


for jloc, row in sel1_sleeplab.iterrows():
    
    try:
        filepath = os.path.join(dir_sleeplab, 'psg_airgo_10hz_' + str(row.study_id).zfill(3) + '.h5')
        data, hdr, fs = read_in_routine(filepath)

        print(f"discordance (%): {row[var_discordance]}")

        dt_start = row['timerange'].split(" - ")[0]
        dt_end = row['timerange'].split(" - ")[1]

        data = data.loc[dt_start : dt_end]
        fig, ax = plt.subplots(1, 1, figsize = (7.2, 3))

        for i_version, staging_version in enumerate(staging_cols_sleeplab):
            ax.plot(data[staging_version] + i_version * 0.04, c=colors[staging_version])
            ax.set_title(f'study_id {row.study_id},sex {row.sex}, age {np.round(row.age)}, ahi {np.round(row.ahi_expert, 1)}, discordance: {np.round(row[var_discordance], 2)}')
            ax.set_yticks([5, 4, 3, 2, 1])
            ax.set_yticklabels(['W', 'R', 'N1', 'N2', 'N3'])

        plt.savefig(os.path.join(plots_savedir, f"sleeplab_{row.study_id}_{dt_start}.png"), dpi=300)
        plt.savefig(os.path.join(plots_savedir, f"sleeplab_{row.study_id}_{dt_start}.png"), dpi=300)

        plt.show()

    except Exception as e:
        print(e)
        continue

discordance (%): 0.009584660862110649


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

discordance (%): 0.010118041797360814


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

discordance (%): 0.014992501050824504


...

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

discordance (%): 0.8614317513027596


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

discordance (%): 0.9492183050539196


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [10]:
plt.close('all')

In [11]:
icu_collection = {}

# selections:
study_id = 15
filepath = os.path.join(dir_icu, 'icusleep_' + str(study_id).zfill(3) + '.h5')
data, hdr, fs = read_in_routine(filepath)
dt_start = '2018-10-14 08:00:00'
dt_end = '2018-10-15 08:00:00'

icu_collection[study_id] = data.loc[dt_start : dt_end, staging_cols_icu]


# selections:
study_id = 29
filepath = os.path.join(dir_icu, 'icusleep_' + str(study_id).zfill(3) + '.h5')
data, hdr, fs = read_in_routine(filepath)
dt_start = '2019-01-24 08:00:00'
dt_end = '2019-01-25 08:00:00'

icu_collection[study_id] = data.loc[dt_start : dt_end, staging_cols_icu]


# selections:
study_id = 186
filepath = os.path.join(dir_icu, 'icusleep_' + str(study_id).zfill(3) + '.h5')
data, hdr, fs = read_in_routine(filepath)
dt_start = '2019-11-14 08:00:00'
dt_end = '2019-11-15 08:00:00'

icu_collection[study_id] = data.loc[dt_start : dt_end, staging_cols_icu]

In [12]:
sleeplab_collection = {}

# selections:
study_id = 26
filepath = os.path.join(dir_sleeplab, 'psg_airgo_10hz_' + str(study_id).zfill(3) + '.h5')
data, hdr, fs = read_in_routine(filepath)
sleeplab_collection[study_id] = data[staging_cols_sleeplab]

# selections:
study_id = 390
filepath = os.path.join(dir_sleeplab, 'psg_airgo_10hz_' + str(study_id).zfill(3) + '.h5')
data, hdr, fs = read_in_routine(filepath)
sleeplab_collection[study_id] = data[staging_cols_sleeplab]

# selections:
study_id = 377
filepath = os.path.join(dir_sleeplab, 'psg_airgo_10hz_' + str(study_id).zfill(3) + '.h5')
data, hdr, fs = read_in_routine(filepath)
sleeplab_collection[study_id] = data[staging_cols_sleeplab]


In [46]:
plt.close('all')
import matplotlib.dates as mdates

width_ratio = 24/7
alpha = 0.75
lw = 1

fig, ax = plt.subplots(3, 2, figsize = (7.2, 5), gridspec_kw={'width_ratios': [1, width_ratio]})

col = 0
for i_subject, study_id in enumerate(sleeplab_collection.keys()):

    data = sleeplab_collection[study_id]
    data = data[data.stage_pred_expert > -1]
    
    for i_version, staging_version in enumerate(staging_cols_sleeplab):
        ax[i_subject, col].plot(data[staging_version] + i_version * 0.05, c=colors[staging_version], lw=lw, alpha=alpha)
#     ax[i_subject, col].set_title(f'study_id {row.study_id}, sex {row.sex}, age {np.round(row.age)} discordance: {np.round(row[var_discordance], 2)}')
    ax[i_subject, col].set_yticks([5, 4, 3, 2, 1])
    ax[i_subject, col].set_yticklabels(['W', 'R', 'N1', 'N2', 'N3'])
    ax[i_subject, col].set_ylim([0.8, 5.2])
    ax[i_subject, col].xaxis.set_major_formatter(mdates.DateFormatter('%H'))
    ax[i_subject, col].xaxis.set_major_locator(mdates.HourLocator(interval=3))

col = 1
for i_subject, study_id in enumerate(icu_collection.keys()):

    data = icu_collection[study_id]
    
    for i_version, staging_version in enumerate(staging_cols_icu):
        ax[i_subject, col].plot(data[staging_version] + i_version * 0.05, c=colors[staging_version], lw=lw, alpha=alpha)
#     ax[i_subject, col].set_title(f'study_id {row.study_id}, sex {row.sex}, age {np.round(row.age)} discordance: {np.round(row[var_discordance], 2)}')
    ax[i_subject, col].set_yticks([5, 4, 3, 2, 1])
    ax[i_subject, col].set_yticklabels(['W', 'R', 'N1', 'N2', 'N3'])
    ax[i_subject, col].set_ylim([0.8, 5.2])
    ax[i_subject, col].xaxis.set_major_formatter(mdates.DateFormatter('%H'))

    
for ax_tmp in ax.flatten():
    ax_tmp.spines["right"].set_visible(False)
    ax_tmp.spines["top"].set_visible(False)

ax[2, 0].set_xlabel('Daytime (hour)')
ax[2, 1].set_xlabel('Daytime (hour)')

ax[0, 0].text(0.5, 1.6, 'Sleeplab (overnight)', ha='center', fontweight='bold', transform=ax[0, 0].transAxes)
ax[0, 1].text(0.5, 1.6, 'ICU (24-hours)', ha='center', fontweight='bold', transform=ax[0, 1].transAxes)

ax[0, 1].text(0, 1.05, '72-year-old male\ndiagnosis: hypotension, shock\nHRV and breathing models concordance 99%', transform=ax[0, 1].transAxes)
ax[1, 1].text(0, 1.05, '69-year-old male\ndiagnosis: gastrointestinal bleeding\nHRV and breathing models concordance 76%', transform=ax[1, 1].transAxes)
ax[2, 1].text(0, 1.05, '79-year-old female\ndiagnosis: hypoxemic respiratory failure, shock\nHRV and breathing models concordance 56% ', transform=ax[2, 1].transAxes)

ax[0, 0].text(0, 1.05, '61-year-old female\nFull night CPAP titration, AHI 3\nHRV-breathing concord. 97%', transform=ax[0, 0].transAxes)
ax[1, 0].text(0, 1.05, '79-year-old female\nSplit Night, AHI D./T. 32/14\nHRV-breathing concord. 86%', transform=ax[1, 0].transAxes) # AHI diagnostic 32, AHI titration 14
ax[2, 0].text(0, 1.05, '76-year-old male\nSplit Night, AHI D./T. 10/6\nHRV-breathing concord. 45%', transform=ax[2, 0].transAxes)

ax[0, 0].legend(['Hypnogram sleep expert', 'Hypnogram HRV-based model', 'Hypnogram breathing-based model'], loc=(0, 1.77), frameon=False, ncol=3)

plt.subplots_adjust(hspace=0.85, wspace=0.17, left=0.04, right=0.99, bottom=0.08, top=0.83)

plt.savefig(os.path.join(plots_savedir, f'hypnograms_ICU_sleeplab_widthr{width_ratio}.jpg'), dpi=600)
plt.savefig(os.path.join(plots_savedir, f'hypnograms_ICU_sleeplab_widthr{width_ratio}.svg'))
plt.savefig(os.path.join(plots_savedir, f'hypnograms_ICU_sleeplab_widthr{width_ratio}.tiff'), dpi=600)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [120]:
ax[0, 0].legend

<bound method Axes.legend of <AxesSubplot:>>

In [ ]:
ax[0, 0].legend

In [85]:
summary_subjects_icu.loc[np.isin(summary_subjects_icu.study_id, [15, 29, 186])]

,inclusion_subject,study_id,mrn,enrolled,age,bmi,sex,osa_diagnosis_yn,osa_diagnosis,icu_mortality,3_month_mortality,icu_los,hospital_los,icu_readmission,diagnosis,lvef,date_of_echo,chf,copd,med_surg,CCI,CCI_weighted,Belt_Length,d1_timerange,d1_hr_mean,d1_hr_std,d1_hr_median,d1_hr_q25,d1_hr_q75,d1_hr_sda,d1_hr_asd,d1_spo2_mean,d1_spo2_std,d1_spo2_median,d1_spo2_q25,d1_spo2_q75,d1_spo2_sda,d1_spo2_asd,d1_spo2_airgo_mean,d1_spo2_airgo_std,d1_spo2_airgo_median,d1_spo2_airgo_q25,d1_spo2_airgo_q75,d1_spo2_airgo_sda,d1_spo2_airgo_asd,d1_spo2_clean_mean,d1_spo2_clean_std,d1_spo2_clean_median,d1_spo2_clean_q25,d1_spo2_clean_q75,d1_spo2_clean_sda,d1_spo2_clean_asd,d1_spo2_airgo_clean_mean,d1_spo2_airgo_clean_std,d1_spo2_airgo_clean_median,d1_spo2_airgo_clean_q25,d1_spo2_airgo_clean_q75,d1_spo2_airgo_clean_sda,d1_spo2_airgo_clean_asd,d1_edw_bp_systolic_mean,d1_edw_bp_systolic_std,d1_edw_bp_systolic_median,d1_edw_bp_systolic_q25,d1_edw_bp_systolic_q75,d1_edw_bp_diastolic_mean,d1_edw_bp_diastolic_std,d1_edw_bp_diastolic_median,d1_edw_bp_diastolic_q25,d1_edw_bp_diastolic_q75,d1_edw_bp_map_mean,d1_edw_bp_map_std,d1_edw_bp_map_median,d1_edw_bp_map_q25,d1_edw_bp_map_q75,d1_cpc_hfc_lfc_ratio_mean,d1_cpc_hfc_lfc_ratio_std,d1_cpc_hfc_lfc_ratio_median,d1_cpc_hfc_lfc_ratio_q25,d1_cpc_hfc_lfc_ratio_q75,d1_cpc_hfc_mean,d1_cpc_hfc_std,d1_cpc_hfc_median,d1_cpc_hfc_q25,d1_cpc_hfc_q75,d1_cpc_hfc_sum,d1_cpc_lfc_mean,d1_cpc_lfc_std,d1_cpc_lfc_median,d1_cpc_lfc_q25,d1_cpc_lfc_q75,d1_cpc_lfc_sum,d1_hrv_nn_mean,d1_hrv_nn_std,d1_hrv_nn_median,d1_hrv_nn_q25,d1_hrv_nn_q75,d1_hrv_nn_sda,d1_hrv_nn_asd,d1_hco3_arterial_mean,d1_hco3_arterial_min,d1_hco3_arterial_max,d1_pco2_arterial_mean,d1_pco2_arterial_min,d1_pco2_arterial_max,d1_po2_arterial_mean,d1_po2_arterial_min,d1_po2_arterial_max,d1_ph_arterial_mean,d1_ph_arterial_min,d1_ph_arterial_max,d1_oxygen_flow_rate_max,d1_sofa_score_mean,d1_sofa_score_min,d1_sofa_score_max,d1_ahi_va_a3,d1_ahi_va_a3_ss,d1_ahi_vb_a3,d1_ahi_vb_a3_ss,d1_ahi_ro_a3,d1_ahi_ro_a3_ss,d1_ahi_rsr_a3,d1_ahi_rsr_a3_ss,d1_ahi_expert,d1_ahi_va_a3_ss_aswti,d1_ahi_vb_a3_ss_aswti,d1_ahi_ro_a3_ss_aswti,d1_ahi_rsr_a3_ss_aswti,d1_hypoxic_burden_va_a3,d1_hypoxic_burden_va_a3_ss,d1_hypoxic_burden_vb_a3,d1_hypoxic_burden_vb_a3_ss,d1_hypoxic_burden_ro_a3,d1_hypoxic_burden_ro_a3_ss,d1_hypoxic_burden_rsr_a3,d1_hypoxic_burden_rsr_a3_ss,d1_hypoxic_burden_expert,d1_hypoxic_burden_va_a3_ss_aswti,d1_hypoxic_burden_vb_a3_ss_aswti,d1_hypoxic_burden_ro_a3_ss_aswti,d1_hypoxic_burden_rsr_a3_ss_aswti,d1_hypoxia_LDI,d1_hypoxia_SDI,d1_hypoxia_TDI,d1_hypoxia_pu90,d1_hypoxia_ss_LDI,d1_hypoxia_ss_SDI,d1_hypoxia_ss_TDI,d1_hypoxia_ss_pu90,d1_hypoxia_ss_aswti_LDI,d1_hypoxia_ss_aswti_SDI,...,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N3_mean,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N3_std,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N3_median,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N3_q25,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N3_q75,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N3_sda,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N3_asd,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N2_mean,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N2_std,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N2_median,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N2_q25,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N2_q75,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N2_sda,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N2_asd,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N1_mean,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N1_std,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N1_median,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N1_q25,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N1_q75,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N1_sda,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N1_asd,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_R_mean,f9_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_R_std,f9_hrv_sd1sd2

In [86]:
summary_subjects_sleeplab.loc[np.isin(summary_subjects_sleeplab.study_id, [26, 390, 387])]

,study_id,mrn,age,sex,cci,matched_all,matched_ahi_0_5,matched_ahi_5_15,matched_ahi_15_30,matched_ahi_30_100,matched_ahi_15_100,n1_timerange,n1_hr_mean,n1_hr_std,n1_hr_median,n1_hr_q25,n1_hr_q75,n1_hr_sda,n1_hr_asd,n1_spo2_mean,n1_spo2_std,n1_spo2_median,n1_spo2_q25,n1_spo2_q75,n1_spo2_sda,n1_spo2_asd,n1_spo2_airgo_mean,n1_spo2_airgo_std,n1_spo2_airgo_median,n1_spo2_airgo_q25,n1_spo2_airgo_q75,n1_spo2_airgo_sda,n1_spo2_airgo_asd,n1_spo2_clean_mean,n1_spo2_clean_std,n1_spo2_clean_median,n1_spo2_clean_q25,n1_spo2_clean_q75,n1_spo2_clean_sda,n1_spo2_clean_asd,n1_spo2_airgo_clean_mean,n1_spo2_airgo_clean_std,n1_spo2_airgo_clean_median,n1_spo2_airgo_clean_q25,n1_spo2_airgo_clean_q75,n1_spo2_airgo_clean_sda,n1_spo2_airgo_clean_asd,n1_edw_bp_systolic_mean,n1_edw_bp_systolic_std,n1_edw_bp_systolic_median,n1_edw_bp_systolic_q25,n1_edw_bp_systolic_q75,n1_edw_bp_diastolic_mean,n1_edw_bp_diastolic_std,n1_edw_bp_diastolic_median,n1_edw_bp_diastolic_q25,n1_edw_bp_diastolic_q75,n1_edw_bp_map_mean,n1_edw_bp_map_std,n1_edw_bp_map_median,n1_edw_bp_map_q25,n1_edw_bp_map_q75,n1_cpc_hfc_lfc_ratio_mean,n1_cpc_hfc_lfc_ratio_std,n1_cpc_hfc_lfc_ratio_median,n1_cpc_hfc_lfc_ratio_q25,n1_cpc_hfc_lfc_ratio_q75,n1_cpc_hfc_mean,n1_cpc_hfc_std,n1_cpc_hfc_median,n1_cpc_hfc_q25,n1_cpc_hfc_q75,n1_cpc_hfc_sum,n1_cpc_lfc_mean,n1_cpc_lfc_std,n1_cpc_lfc_median,n1_cpc_lfc_q25,n1_cpc_lfc_q75,n1_cpc_lfc_sum,n1_hrv_nn_mean,n1_hrv_nn_std,n1_hrv_nn_median,n1_hrv_nn_q25,n1_hrv_nn_q75,n1_hrv_nn_sda,n1_hrv_nn_asd,n1_hco3_arterial_mean,n1_hco3_arterial_min,n1_hco3_arterial_max,n1_pco2_arterial_mean,n1_pco2_arterial_min,n1_pco2_arterial_max,n1_po2_arterial_mean,n1_po2_arterial_min,n1_po2_arterial_max,n1_ph_arterial_mean,n1_ph_arterial_min,n1_ph_arterial_max,n1_oxygen_flow_rate_max,n1_sofa_score_mean,n1_sofa_score_min,n1_sofa_score_max,n1_ahi_va_a3,n1_ahi_va_a3_ss,n1_ahi_vb_a3,n1_ahi_vb_a3_ss,n1_ahi_ro_a3,n1_ahi_ro_a3_ss,n1_ahi_rsr_a3,n1_ahi_rsr_a3_ss,n1_ahi_expert,n1_ahi_va_a3_ss_aswti,n1_ahi_vb_a3_ss_aswti,n1_ahi_ro_a3_ss_aswti,n1_ahi_rsr_a3_ss_aswti,n1_hypoxic_burden_va_a3,n1_hypoxic_burden_va_a3_ss,n1_hypoxic_burden_vb_a3,n1_hypoxic_burden_vb_a3_ss,n1_hypoxic_burden_ro_a3,n1_hypoxic_burden_ro_a3_ss,n1_hypoxic_burden_rsr_a3,n1_hypoxic_burden_rsr_a3_ss,n1_hypoxic_burden_expert,n1_hypoxic_burden_va_a3_ss_aswti,n1_hypoxic_burden_vb_a3_ss_aswti,n1_hypoxic_burden_ro_a3_ss_aswti,n1_hypoxic_burden_rsr_a3_ss_aswti,n1_hypoxia_LDI,n1_hypoxia_SDI,n1_hypoxia_TDI,n1_hypoxia_pu90,n1_hypoxia_ss_LDI,n1_hypoxia_ss_SDI,n1_hypoxia_ss_TDI,n1_hypoxia_ss_pu90,n1_hypoxia_ss_aswti_LDI,n1_hypoxia_ss_aswti_SDI,n1_hypoxia_ss_aswti_TDI,n1_hypoxia_ss_aswti_pu90,n1_self_similarity_sleep,n1_self_similarity_sleep_q50,n1_self_similarity_sleep_q75,n1_self_similarity_apnea,n1_self_similarity_apnea_q50,n1_self_similarity_apnea_q75,n1_opioids_sum,n1_benzos_sum,n1_antipsychotics_sum,n1_dex_studydrug_sum,...,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N3_mean,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N3_std,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N3_median,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N3_q25,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N3_q75,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N3_sda,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N3_asd,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N2_mean,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N2_std,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N2_median,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N2_q25,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N2_q75,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N2_sda,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N2_asd,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N1_mean,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N1_std,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N1_median,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N1_q25,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N1_q75,n1_hrv_sd1sd2_stagewise_agrelaxed_amendsumeffort_N


MRN: 2964631.0  
Air026  
/media/mad3/Datasets_ConvertedData/sleeplab/natus_data/Burgess~ Chery_46639851-2530-4db3-b4fd-0c5e8dca3ec2  

MRN: 2560550.0  
Air377  
/media/mad3/Datasets_ConvertedData/sleeplab/natus_data/Snyder~ AmyBet_67bfad41-f8fc-4109-876c-6ee0c9995a44  

MRN: 1864323.0  
Air387  
/media/mad3/Datasets_ConvertedData/sleeplab/natus_data/Davis~ Lawrenc_30f7bb7f-8ab9-4207-8c15-13e6d7f6514a  
